In [1]:
import pandas as pd

from control_totals.util import Pipeline

p = Pipeline('configs')

In [38]:
with pd.HDFStore('data/pipeline.h5') as store:
    tables = store.keys()
tables

['/adjusted_emp_change_targets',
 '/adjusted_emp_change_targets_no_res_con',
 '/adjusted_emp_change_targets_res_con',
 '/adjusted_king_targets',
 '/adjusted_total_pop_change_targets',
 '/adjusted_units_change_targets',
 '/block_control_area_xwalk',
 '/blocks',
 '/control_areas',
 '/control_target_xwalk',
 '/county',
 '/dec_block_data',
 '/decennial_by_control_area',
 '/employment_2018_by_control_area',
 '/employment_2019_by_control_area',
 '/employment_2020_by_control_area',
 '/employment_2023_by_control_area',
 '/extrapolated_targets',
 '/hct_buffers',
 '/hct_stops',
 '/king_targets',
 '/kitsap_targets',
 '/military_bases',
 '/military_bases_xwalk',
 '/national_forest',
 '/national_park',
 '/natural_resource',
 '/ofm_parcel_control_area_xwalk',
 '/ofm_parcelized_2019',
 '/ofm_parcelized_2019_by_control_area',
 '/ofm_parcelized_2020',
 '/ofm_parcelized_2020_by_control_area',
 '/ofm_parcelized_2023',
 '/ofm_parcelized_2023_by_control_area',
 '/old_control_areas',
 '/parcel_pts_current',

In [4]:
p.get_table('split_hct_base_data_2020')

,split_geo_id,nosplit_geo_id,households,persons,jobs,name,RGID
0,1,1,40311.0,94752.0,52482.0,Bellevue,1
1,2,2,110298.0,268967.0,164441.0,Seattle,1
2,3,3,19390.0,47634.0,38681.0,Auburn,2
3,4,4,9049.0,21009.0,36376.0,Bothell,2
4,5,5,16439.0,38785.0,21077.0,Burien,2
...,...,...,...,...,...,...,...
225,1169,169,533.0,1395.0,87.0,Silver Firs Gap,5
226,1176,176,12.0,35.0,2.0,Snohomish Rural,6
227,1301,301,1.0,1.0,0.0,King Rural Ag & Resource,6
228,1401,401,11.0,20.0,3.0,NBK - Bremerton,1


In [77]:
from control_totals.steps.extrapolate_to_controls_year import maybe_load_adjusted_targets

In [78]:
target_frames = []

pop_change_targets = maybe_load_adjusted_targets(
    p, 'adjusted_total_pop_change_targets', 'total_pop_chg'
)
if pop_change_targets is not None:
    target_frames.append(pop_change_targets)

In [79]:
unit_change_targets = maybe_load_adjusted_targets(
    p, 'adjusted_units_change_targets', 'unit_chg'
)
if unit_change_targets is not None:
    target_frames.append(unit_change_targets)

In [81]:




# if king county method is specified, bring in those targets as well
if p.settings['target_types']['king_cnty_method']:
    king_targets = p.get_table('adjusted_king_targets')
    target_frames.append(king_targets)

if target_frames:
    pop_unit_targets = pd.concat(target_frames, ignore_index=True)
    if 'start' in pop_unit_targets.columns:
        pop_unit_targets = pop_unit_targets.drop(columns=['start'])
else:
    pop_unit_targets = pd.DataFrame()

emp_change_targets_res_con = p.get_table('adjusted_emp_change_targets_res_con')
emp_change_targets_no_res_con = p.get_table('adjusted_emp_change_targets_no_res_con')
emp_targets = pd.concat([emp_change_targets_res_con, emp_change_targets_no_res_con], ignore_index=True).drop(columns=['start'])
df_out = pop_unit_targets.merge(emp_targets.drop(columns=['county_id','rgid'], errors='ignore'), on='target_id', how='outer')

In [82]:
df_out

,target_id,total_pop_chg,total_pop_chg_adj,units_chg,units_chg_adj,rgid,county_id,ofm_total_pop,ofm_hhpop,ofm_units,...,Emp_TotNoMil_2023,Emp_ConRes_2023,TotEmpNoMil-ResCon_2023,emp_2044,res_con_pct,res_con_target_pct,emp_chg_cnty_sum,res_con_emp_chg_cnty_sum,res_con_emp_chg_target,emp_chg_adj_res_con
0,1,NaN,NaN,35000.0,31505.0,1,53033,NaN,NaN,NaN,...,164447,8984,155464,231230,0.054632,3824.210840,490831.0,13743.268,1649.241302,66783.241302
1,2,NaN,NaN,112000.0,75799.0,1,53033,NaN,NaN,NaN,...,710890,33336,677554,848911,0.046893,7948.419587,490831.0,13743.268,3427.860654,138020.860654
2,3,NaN,NaN,12000.0,11032.0,2,53033,NaN,NaN,NaN,...,52024,6409,45615,71170,0.123193,2404.730125,490831.0,13743.268,1037.071544,19146.071544
3,4,NaN,NaN,5800.0,4973.0,2,53033,NaN,NaN,NaN,...,19103,1118,17986,27785,0.058525,555.985971,490831.0,13743.268,239.776274,8681.776274
4,5,NaN,NaN,7500.0,7162.0,2,53033,NaN,NaN,NaN,...,14694,1444,13250,19253,0.098271,468.754594,490831.0,13743.268,202.156594,4559.156594
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,170,405.0,402.0,203.0,202.0,5,53061,1393.035266,1391.035266,519.790294,...,417,98,319,412,0.235012,8.695444,171820.0,4810.960,2.928730,-5.071270
134,172,290.0,290.0,138.0,138.0,5,53061,142.028000,132.028000,53.954001,...,258,0,258,735,0.000000,0.000000,171820.0,4810.960,0.000000,477.000000
135,174,149.0,149.0,73.0,73.0,5,53061,38.075999,38.075999,15.997000,...,0,0,0,1,0.000000,0.000000,171820.0,4810.960,0.000000,1.000000
136,175,271.0,271.0,140.0,140.0,5,53061,0.000000,0.000000,0.000000,...,4,0,4,32,0.000000,0.000000,171820.0,4810.960,0.000000,28.000000


In [83]:
col = 'hh'
df = df_out.copy()
base_year = p.settings['base_year']
targets_end_year = p.settings['targets_end_year']
controls_end_year = p.settings['end_year']

# extrapolate to controls horizon year based on the avg annual change from base year to targets horizon year
years_to_target = targets_end_year - base_year
years_to_control = controls_end_year - base_year
annual_change_col = f'{col}_annual_change'
if col == 'emp':
    base_col = f'Emp_TotNoMil_{base_year}'
else:
    base_col = f'dec_{col}'
target_col = f'{col}_{targets_end_year}'
df[annual_change_col] = (df[target_col] - df[base_col]) / years_to_target
control_col = f'{col}_{controls_end_year}'
df[control_col] = df[base_col] + df[annual_change_col] * years_to_control
df[control_col] = df[control_col].round(0).fillna(0).astype(int)

In [93]:
df[['dec_hh','ofm_hh','hh_2044', 'hh_annual_change']]

,dec_hh,ofm_hh,hh_2044,hh_annual_change
0,60953.0,NaN,91672.000000,1462.809524
1,345046.0,NaN,422559.000000,3691.095238
2,27057.0,NaN,37362.000000,490.714286
3,12006.0,NaN,16878.000000,232.000000
4,19874.0,NaN,26717.000000,325.857143
...,...,...,...,...
133,NaN,493.364029,694.702262,NaN
134,NaN,51.875000,184.750169,NaN
135,NaN,15.970000,85.657036,NaN
136,NaN,0.000000,134.745947,NaN


In [87]:
df.loc[df.hh_annual_change.isna()]

,target_id,total_pop_chg,total_pop_chg_adj,units_chg,units_chg_adj,rgid,county_id,ofm_total_pop,ofm_hhpop,ofm_units,...,TotEmpNoMil-ResCon_2023,emp_2044,res_con_pct,res_con_target_pct,emp_chg_cnty_sum,res_con_emp_chg_cnty_sum,res_con_emp_chg_target,emp_chg_adj_res_con,hh_annual_change,hh_2050
56,65,20252.0,18734.0,9556.0,8505.0,1,53035,41067.368858,38503.033836,18539.043445,...,35718,49061,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
57,67,2762.0,2695.0,1227.0,1198.0,1,53035,10137.093894,10132.093894,4412.460440,...,1324,3903,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
58,68,9896.0,9279.0,4823.0,4552.0,2,53035,19641.608507,19430.608507,8416.777713,...,13937,24797,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
59,69,4524.0,4172.0,1977.0,1793.0,3,53035,24980.608008,24777.608008,11347.393153,...,8658,11183,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
60,70,3200.0,3082.0,1519.0,1474.0,3,53035,2553.340020,2553.340020,1218.213981,...,1168,2489,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,170,405.0,402.0,203.0,202.0,5,53061,1393.035266,1391.035266,519.790294,...,319,412,0.235012,8.695444,171820.0,4810.96,2.928730,-5.071270,NaN,0
134,172,290.0,290.0,138.0,138.0,5,53061,142.028000,132.028000,53.954001,...,258,735,0.000000,0.000000,171820.0,4810.96,0.000000,477.000000,NaN,0
135,174,149.0,149.0,73.0,73.0,5,53061,38.075999,38.075999,15.997000,...,0,1,0.000000,0.000000,171820.0,4810.96,0.000000,1.000000,NaN,0
136,175,271.0,271.0,140.0,140.0,5,53061,0.000000,0.000000,0.000000,...,4,32,0.000000,0.000000,171820.0,4810.96,0.000000,28.000000,NaN,0
